In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F

In [4]:
train_df = pd.read_csv("/content/train.csv")

Setup (Run Before Attempting Questions) :

Use the following two fine-tuned sequence classification checkpoints:

DeBERTa: microsoft/deberta-v3-small (fine-tuned checkpoint)

RoBERTa: roberta-base (fine-tuned checkpoint)

In [87]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
DEBERTA_MODEL = "microsoft/deberta-v3-small"
ROBERTA_MODEL = "FacebookAI/roberta-base"

deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_MODEL)
deberta_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_MODEL, num_labels = 1)

roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_MODEL)
roberta_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_MODEL, num_labels = 1)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.de

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Load the fine-tuned DeBERTa and RoBERTa models.

For the prompt at row index 25, perform inference using each model independently and apply Softmax to obtain class probabilities.

Question 1:

Which answer option receives the highest probability from the DeBERTa model, and what is that probability?

*
(answer format : eg - A, probability of A)

In [88]:
label_map = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

row = train_df.iloc[25]
prompt = row["prompt"]
options = [row[option] for option in label_map.values()]

texts = [
    prompt + " [SEP] " + option
    for option in options
]

print(len(texts))
print(texts)

5
["Choose the correct answer: What is Hesse's principle of transfer in geometry? carefully. [SEP] Hesse's principle of transfer is a concept in biology that explains the transfer of genetic information from one generation to another.", "Choose the correct answer: What is Hesse's principle of transfer in geometry? carefully. [SEP] Hesse's principle of transfer is a concept in chemistry that explains the transfer of electrons between atoms in a chemical reaction.", "Choose the correct answer: What is Hesse's principle of transfer in geometry? carefully. [SEP] Hesse's principle of transfer is a concept in physics that explains the transfer of energy from one object to another.", "Choose the correct answer: What is Hesse's principle of transfer in geometry? carefully. [SEP] Hesse's principle of transfer is a concept in economics that explains the transfer of wealth from one individual to another.", "Choose the correct answer: What is Hesse's principle of transfer in geometry? carefully. [

In [97]:
deberta_model.eval()

inputs = deberta_tokenizer(
    texts,
    padding = True,
    truncation = True,
    max_length = 256,
    return_tensors="pt"
)

with torch.no_grad():
  outputs = deberta_model(**inputs)

logits = outputs.logits.squeeze(-1)
print(logits)

probs_deberta = F.softmax(logits, dim=-1)

print(probs_deberta)

predicted_label = torch.argmax(probs_deberta)

tensor([-0.2864, -0.2864, -0.2861, -0.2861, -0.2856], dtype=torch.float16)
tensor([0.2000, 0.2000, 0.2000, 0.2000, 0.2001], dtype=torch.float16)


In [99]:
print(f"{label_map[predicted_label.item()]}, {probs_deberta[predicted_label.item()]:.4f}")

E, 0.2001


In [101]:
roberta_model.eval()

inputs = roberta_tokenizer(
    texts,
    padding = True,
    max_length = 256,
    truncation = True,
    return_tensors = "pt"
)

with torch.no_grad():
  outputs = roberta_model(**inputs)

logits = outputs.logits.squeeze(-1)

probs_roberta = F.softmax(logits, dim = -1)

print(probs_roberta)

predicted_label = torch.argmax(probs_roberta)

tensor([0.1979, 0.1997, 0.1996, 0.2002, 0.2027])


In [103]:
print(f"{label_map[predicted_label.item()]}, {probs_roberta[predicted_label.item()]:.4f}")

E, 0.2027


Using the same sample (row index 25), average the class probabilities from both models.

Average Probability = [P(DeBERTa) + P(RoBERTa)]/2

Question 2:

Which answer option receives the highest averaged probability after simple probability ensembling?

In [104]:
average_prob = (probs_deberta + probs_roberta)/2
print(average_prob)

predicted_label = torch.argmax(average_prob).item()

print(f"{label_map[predicted_label]}, {average_prob[predicted_label]:.5f}")

tensor([0.1989, 0.1998, 0.1998, 0.2001, 0.2014])
E, 0.20140


Apply weighted probability averaging after Softmax using the following weights:

DeBERTa: 0.70

RoBERTa: 0.30

Compute:

P(final) = [0.7 × P(DeBERTa)] + [0.3 × P(RoBERTa)]

Question 3:

Which answer option is ranked first after weighted ensembling?

In [105]:
weighted_probs = 0.70*probs_deberta + 0.30*probs_roberta
print(weighted_probs)

predicted_label = torch.argmax(weighted_probs).item()

print(f"{label_map[predicted_label]}, {weighted_probs[predicted_label]:.5f}")

tensor([0.1994, 0.1999, 0.1999, 0.2001, 0.2008])
E, 0.20083


In [112]:
ranked_probs = weighted_probs.sort(descending=True)
print(ranked_probs)

top3 = ""

for item in ranked_probs[1]:
  top3 += label_map[item.item()]
  if(len(top3) == 3):
    break

print(f"Top 3 Prediction string of row index 25 is:\n{top3[0]} {top3[1]} {top3[2]}")

torch.return_types.sort(
values=tensor([0.2008, 0.2001, 0.1999, 0.1999, 0.1994]),
indices=tensor([4, 3, 1, 2, 0]))
Top 3 Prediction string of row index 25 is:
E D B


Run the weighted ensemble pipeline on every row of test.csv.

Save the predictions in a file named submission.csv using the required Kaggle format:

id,prediction

where the prediction column contains the Top-3 ranked options separated by spaces.

Question 5:

Exactly how many prediction rows are present in the generated file (excluding the header)?

In [59]:
test_df = pd.read_csv("/content/test.csv")

In [114]:
from tqdm.notebook import tqdm

predictions = []

for index,row in tqdm(test_df.iterrows(),total=len(test_df), desc="Running Inference:"):
  prompt = row["prompt"]

  options = [row[option] for option in label_map.values()]

  texts = [
      prompt + " [SEP] " + option
      for option in options
  ]

  input_roberta = roberta_tokenizer(
      texts,
      truncation = True,
      padding = True,
      max_length = 256,
      return_tensors = "pt"
  )

  input_deberta = deberta_tokenizer(
      texts,
      truncation = True,
      padding = True,
      max_length = 256,
      return_tensors = "pt"
  )

  with torch.no_grad():
    roberta_output = roberta_model(**input_roberta)
    deberta_output = deberta_model(**input_deberta)

  roberta_probs = F.softmax(roberta_output.logits.squeeze(-1), dim=-1)
  deberta_probs = F.softmax(deberta_output.logits.squeeze(-1), dim=-1)

  weighted_probs = 0.7*deberta_probs + 0.3*roberta_probs
  print(weighted_probs)
  top3 = ""
  weighted_probs_sorted = weighted_probs.sort(descending = True)

  for item in weighted_probs_sorted[1]:
    top3 += label_map[item.item()]
    if(len(top3) == 3):
      break

  prediction = f"{top3[0]} {top3[1]} {top3[2]}"
  print(prediction)

  predictions.append(prediction)

Running Inference::   0%|          | 0/500 [00:00<?, ?it/s]

['Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics? carefully. [SEP] For every eigenstate of one Hamiltonian, its partner Hamiltonian has a corresponding eigenstate with the same energy.', 'Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics? carefully. [SEP] For every eigenstate of one Hamiltonian, its partner Hamiltonian has a corresponding eigenstate with a higher energy.', 'Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics? carefully. [SEP] For every eigenstate of one Hamiltonian, its partner Hamiltonian has a corresponding eigenstate with a different spin.', 'Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics? carefully. [SEP] For every eigenstate of one Ha

In [68]:
submission_df = pd.DataFrame({
    "id": test_df["id"],
    "prediction": predictions
})

submission_df.head()

,index,prediction
0,0,A B C
1,1,A B C
2,2,A B C
3,3,A B C
4,4,A B C
...,...,...
495,495,A B C
496,496,A B C
497,497,A B C
498,498,A B C


For the first 50 rows of test.csv, create two versions of every prompt:

1.Original prompt

2.Instruction-augmented prompt by prepending: "Answer the following multiple-choice question carefully:"

Run inference using DeBERTa on both versions.

Average the predicted probabilities from both passes.

Question 6:

How many of the first 50 rows produce a different Top-1 prediction after applying Test-Time Augmentation?

In [117]:
questions = 50
changed = 0
for index,row in tqdm(test_df.iterrows(), total = questions, desc="Running Inference:"):
  if index == questions:
    break

  prompt = row["prompt"]

  options = [row[option] for option in label_map.values()]

  original_text = [
      prompt + " [SEP] " + option
      for option in options
  ]

  augment_text = [
      "Answer the following multiple-choice question carefully: " + prompt + " [SEP] " + option
      for option in options
  ]

  original_inputs = deberta_tokenizer(
        original_text,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

  augmented_inputs = deberta_tokenizer(
      augment_text,
      padding=True,
      truncation=True,
      max_length=256,
      return_tensors="pt"
  )

  with torch.no_grad():
    original_logits = deberta_model(**original_inputs).logits.squeeze(-1)
    augmented_logits = deberta_model(**augmented_inputs).logits.squeeze(-1)

  original_probs = F.softmax(original_logits, dim=0)
  augmented_probs = F.softmax(augmented_logits, dim=0)

  avg_probs = (original_probs + augmented_probs) / 2

  original_pred = torch.argmax(original_probs).item()
  tta_pred = torch.argmax(avg_probs).item()

  if original_pred != tta_pred:
    changed += 1

print("Number of changed Top-1 predictions:", changed)

Running Inference::   0%|          | 0/50 [00:00<?, ?it/s]

Number of changed Top-1 predictions: 9


Process the first 100 rows of test.csv. And compare the Top-1 prediction from:

1. DeBERTa

2. Weighted Ensemble

Question 7:

How many rows have different Top-1 predictions?

In [118]:
questions = 100
changed = 0

In [120]:
from tqdm.notebook import tqdm

predictions = []

for index,row in tqdm(test_df.head(questions).iterrows(),total=questions, desc="Running Inference:"):
  prompt = row["prompt"]

  options = [row[option] for option in label_map.values()]

  texts = [
      prompt + " [SEP] " + option
      for option in options
  ]

  input_roberta = roberta_tokenizer(
      texts,
      truncation = True,
      padding = True,
      max_length = 256,
      return_tensors = "pt"
  )

  input_deberta = deberta_tokenizer(
      texts,
      truncation = True,
      padding = True,
      max_length = 256,
      return_tensors = "pt"
  )

  with torch.no_grad():
    roberta_output = roberta_model(**input_roberta)
    deberta_output = deberta_model(**input_deberta)

  roberta_probs = F.softmax(roberta_output.logits.squeeze(-1), dim=-1)
  deberta_probs = F.softmax(deberta_output.logits.squeeze(-1), dim=-1)

  weighted_probs = 0.7*deberta_probs + 0.3*roberta_probs

  deberta_pred = torch.argmax(deberta_probs).item()
  weighted_pred = torch.argmax(weighted_probs).item()

  if deberta_pred != weighted_pred:
    changed += 1


print("Number of changed Top-1 predictions:", changed)

Running Inference::   0%|          | 0/100 [00:00<?, ?it/s]

Number of changed Top-1 predictions: 53


For the first 100 rows of test.csv, record the highest class probability (confidence) predicted by:

1. DeBERTa

2. Weighted Ensemble

For every row, compute:

Confidence Gain = Ensemble Confidence−DeBERTa Confidence

Question 8:

How many rows have a positive confidence gain (greater than 0)?

In [122]:
positive_confidence_gain = 0

In [124]:
from tqdm.notebook import tqdm

predictions = []

for index,row in tqdm(test_df.head(questions).iterrows(),total=questions, desc="Running Inference:"):
  prompt = row["prompt"]

  options = [row[option] for option in label_map.values()]

  texts = [
      prompt + " [SEP] " + option
      for option in options
  ]

  input_roberta = roberta_tokenizer(
      texts,
      truncation = True,
      padding = True,
      max_length = 256,
      return_tensors = "pt"
  )

  input_deberta = deberta_tokenizer(
      texts,
      truncation = True,
      padding = True,
      max_length = 256,
      return_tensors = "pt"
  )

  with torch.no_grad():
    roberta_output = roberta_model(**input_roberta)
    deberta_output = deberta_model(**input_deberta)

  roberta_probs = F.softmax(roberta_output.logits.squeeze(-1), dim=-1)
  deberta_probs = F.softmax(deberta_output.logits.squeeze(-1), dim=-1)

  weighted_probs = 0.7*deberta_probs + 0.3*roberta_probs

  deberta_confidence = torch.max(deberta_probs).item()
  ensemble_confidence = torch.max(weighted_probs).item()

  confidence_gain = ensemble_confidence - deberta_confidence

  if confidence_gain > 0:
    positive_confidence_gain += 1

print("Rows with positive confidence gain:", positive_confidence_gain)

Running Inference::   0%|          | 0/100 [00:00<?, ?it/s]

Rows with positive confidence gain: 126


For the first 100 rows of test.csv, compare the Top-3 prediction strings generated by:

1. DeBERTa alone

2. Weighted Ensemble

Question 9:

How many rows have at least one change in their ordered Top-3 ranking after ensembling?

*
Examples:

A C D vs. A D C

In [125]:
question = 100
prediction_change = 0

In [126]:
from tqdm.notebook import tqdm

predictions = []

for index,row in tqdm(test_df.head(questions).iterrows(),total=questions, desc="Running Inference:"):
  prompt = row["prompt"]

  options = [row[option] for option in label_map.values()]

  texts = [
      prompt + " [SEP] " + option
      for option in options
  ]

  input_roberta = roberta_tokenizer(
      texts,
      truncation = True,
      padding = True,
      max_length = 256,
      return_tensors = "pt"
  )

  input_deberta = deberta_tokenizer(
      texts,
      truncation = True,
      padding = True,
      max_length = 256,
      return_tensors = "pt"
  )

  with torch.no_grad():
    roberta_output = roberta_model(**input_roberta)
    deberta_output = deberta_model(**input_deberta)

  roberta_probs = F.softmax(roberta_output.logits.squeeze(-1), dim=-1)
  deberta_probs = F.softmax(deberta_output.logits.squeeze(-1), dim=-1)

  weighted_probs = 0.7*deberta_probs + 0.3*roberta_probs

  top3_weighted = ""
  top3_deberta  = ""
  weighted_probs_sorted = weighted_probs.sort(descending = True)
  deberta_probs_sorted = deberta_probs.sort(descending = True)

  for item in weighted_probs_sorted[1]:
    top3_weighted += label_map[item.item()]
    if(len(top3_weighted) == 3):
      break

  for item in deberta_probs_sorted[1]:
    top3_deberta += label_map[item.item()]
    if(len(top3_deberta) == 3):
      break

  if top3_weighted != top3_deberta:
    prediction_change += 1

print(f"Rows with at least one change in ordered top 3: {prediction_change}")

Running Inference::   0%|          | 0/100 [00:00<?, ?it/s]

Rows with at least one change in ordered top 3: 93


Using the Top-3 predictions generated by your weighted ensemble for the first 100 validation samples, compute the MAP@3 score.

Question 10:


What is the final MAP@3 score?

(Round to 4 decimal places.)

In [127]:
map3_score = 0

In [128]:
def map3(prediction, answer):
  items = prediction.split()

  if answer == items[0]:
    return 1.0
  elif answer == items[1]:
    return 0.5
  elif answer == items[2]:
    return 1/3
  else:
    return 0.0

In [129]:
from tqdm.notebook import tqdm

predictions = []

for index,row in tqdm(train_df.head(100).iterrows(),total=questions, desc="Running Inference:"):
  prompt = row["prompt"]

  answer = row["answer"]

  options = [row[option] for option in label_map.values()]

  texts = [
      prompt + " [SEP] " + option
      for option in options
  ]

  input_roberta = roberta_tokenizer(
      texts,
      truncation = True,
      padding = True,
      max_length = 256,
      return_tensors = "pt"
  )

  input_deberta = deberta_tokenizer(
      texts,
      truncation = True,
      padding = True,
      max_length = 256,
      return_tensors = "pt"
  )

  with torch.no_grad():
    roberta_output = roberta_model(**input_roberta)
    deberta_output = deberta_model(**input_deberta)

  roberta_probs = F.softmax(roberta_output.logits.squeeze(-1), dim=-1)
  deberta_probs = F.softmax(deberta_output.logits.squeeze(-1), dim=-1)

  weighted_probs = 0.7*deberta_probs + 0.3*roberta_probs

  top3_weighted = ""
  weighted_probs_sorted = weighted_probs.sort(descending = True)

  for item in weighted_probs_sorted[1]:
    top3_weighted += label_map[item.item()]
    if(len(top3_weighted) == 3):
      break

  prediction = f'{top3_weighted[0]} {top3_weighted[1]} {top3_weighted[2]}'
  map3_score += map3(prediction, answer)

print(f'Map Score : {map3_score/100:.4f}')


Running Inference::   0%|          | 0/100 [00:00<?, ?it/s]

Map Score : 0.5233
